
# ML Failure Investigation Lab
## Thinking Like an ML Engineer

### Goal

In this workshop, you are NOT evaluated on:
- highest accuracy
- fastest implementation
- copying models

You ARE evaluated on:
- reasoning
- diagnosis
- justification
- investigation
- error analysis
- evidence-based improvements

---

## Rules

You must justify every decision.

For every modification:
1. What problem are you trying to solve?
2. What evidence suggests this problem exists?
3. Why should your solution help?
4. What tradeoff might it introduce?

---

## Main Question

> "Why is the model failing, and how can we prove it?"



# Dataset

We will use the Credit Card Fraud Detection dataset.

Download manually from:

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

Place `creditcard.csv` in the same folder as this notebook.


In [1]:
!pip install kagglehub

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Path to dataset files: /kaggle/input/creditcardfraud


In [3]:
ls -l $path

total 147296
-rw-r--r-- 1 1000 1000 150828752 Dec 30 03:50 creditcard.csv


In [7]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt



# Task 1 — Load and Investigate the Dataset

Without training any model yet:

Answer:
1. How many rows and columns exist? **5 rows 31 Columns**
2. Are there missing values? **NO**
3. Which column is the target? **Class Column 0:not Fraud, 1:Fraud**
4. Is the dataset balanced? **NO, 99.8% for 0 and 0.2% for 1**
5. What immediate risks do you notice?

**- A naive model that predicts "not fraud" for every sample would achieve ~99.83% accuracy — completely useless for fraud detection.**

**- Standard metrics like accuracy are misleading.**

**- The minority class (fraud) might be underrepresented in either the train or test split if stratification is not used.**

---

## Reasoning Questions

1. Why might class imbalance be dangerous?

**Because the learning algorithm sees more examples from the majority class, it learns to bias its predictions toward that class. It can minimize the overall loss by nearly always predicting "not fraud," which is statistically rewarded even though it completely fails the real objective.**

2. Why can accuracy become misleading?

**Accuracy counts ALL correct predictions equally. In a 99.83%/0.17% split, predicting "not fraud" for every sample gives 99.83% accuracy — yet the model catches zero fraud cases. The metric hides the catastrophic failure on the minority class.**

3. If fraud cases are very rare, what metric may matter more?

**Recall** (sensitivity / True Positive Rate) is the most critical metric. It measures the fraction of actual fraud cases that the model correctly identifies. Missing a real fraud case (False Negative) is far more costly than a false alarm (False Positive) in banking. The **F1-score** and **Precision-Recall AUC** are also more informative than accuracy here.

In [8]:

df = pd.read_csv(os.path.join(path, "creditcard.csv"))
df.head()



,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [10]:

df.info()

print(df.isnull().sum())

df["Class"].value_counts()

df["Class"].value_counts(normalize=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

,proportion
Class,
0,0.998273
1,0.001727



# Accuracy Trap

Suppose:
- 99.8% of transactions are NOT fraud

A model predicting:
> "Not Fraud" for every sample

could still achieve extremely high accuracy.

---

## Formula

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

---

## Think Carefully

Would this model actually be useful?

Why or why not?

**No, this model is completely useless** despite high accuracy.

- The entire value of a fraud detection system is catching fraud cases (Class = 1).
- A model that always predicts "not fraud" achieves ~99.83% accuracy while having **Recall = 0%** for the fraud class — it never flags a single fraudulent transaction.
- In banking, every missed fraud case (False Negative) means real financial loss for the customer/bank.



# Task 2 — Train/Test Split

Questions:
1. Why do we use stratify?

`stratify=y` ensures that the **proportion of each class is preserved** in both the training set and the test set. Without it, random splitting on a highly imbalanced dataset could, by chance, place most or all of the rare fraud samples in one split — leaving the other set nearly fraud-free.

2. What risk happens if fraud samples are distributed unevenly?

- If most fraud samples end up in training only: the model sees insufficient real-world test examples → evaluation is unreliable.
- If most fraud samples end up in testing only: the model trains on almost no fraud examples → it learns nothing useful about fraud.
- Either way, the evaluation would be **unrepresentative** and the model would be deployed with a false sense of performance.

In [11]:

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



# Task 3 — Scaling Decision

We will scale features before Logistic Regression.

## Questions

1. Why might scaling matter for Logistic Regression?

Logistic Regression optimizes a loss function using gradient-based methods. Features with large magnitudes (e.g., `Amount` can be thousands, while V1–V28 are PCA-transformed and near zero) dominate the gradient updates. This can cause:
- Slow or unstable convergence
- The optimizer spending disproportionate effort on large-scale features
- Coefficients being poorly calibrated

Standardizing features (mean=0, std=1) ensures all features contribute equally during gradient descent.

2. Would scaling matter equally for Decision Trees?

**No.** Decision trees (and Random Forest, XGBoost, etc.) make splits based on **rank order** of values, not their absolute magnitude. `Amount = 5000` is just "greater than split threshold X" — the actual scale doesn't affect where the split happens. These tree-based models are **scale-invariant**.

3. Which algorithms are sensitive to feature magnitude?

- Logistic Regression ✅
- Support Vector Machines (SVM) ✅
- K-Nearest Neighbors (KNN) ✅
- Neural Networks ✅
- Principal Component Analysis (PCA) ✅
- Ridge/Lasso regression ✅

**Scale-invariant algorithms:**
- Decision Trees ❌ (no scaling needed)
- Random Forest ❌
- Gradient Boosting (XGBoost, LightGBM) ❌
- Naive Bayes ❌

In [12]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



# Task 4 — Baseline Model

Train a Logistic Regression model WITHOUT solving imbalance.

Before running:
- Predict what might happen
- Which metric may look deceptively good?

**Expected outcome:**
- **Accuracy** will look very high (~99%+) — deceptively good.
- **Recall for fraud (Class=1)** will likely be very low — the model will barely detect any fraud.
- The model is biased toward predicting "0" (legitimate) because that class dominates the training data.

**Why?** Without addressing imbalance, Logistic Regression minimizes overall loss by correctly classifying the majority class and largely ignoring the minority class.

In [13]:

model = LogisticRegression()

model.fit(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)


In [14]:

print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))
print("F1:", f1_score(y_test, preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))


Accuracy: 0.9991397773954567
Precision: 0.8266666666666667
Recall: 0.6326530612244898
F1: 0.7167630057803468

Confusion Matrix
[[56851    13]
 [   36    62]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.63      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962




# Task 5 — Error Analysis

Analyze the results carefully.

---

## Questions

1. Is the model actually good?

**No.** Despite high accuracy (~99%), the model fails at its core objective: detecting fraud. The high accuracy is purely a result of correctly labeling the overwhelming majority of legitimate transactions — not because it learned to detect fraud.

2. Which metric is most concerning?

**Recall for Class 1 (fraud).** The baseline model likely achieves very low recall on fraud (potentially 50–65%). This means it misses a substantial fraction of actual fraud cases.

3. What type of errors are dangerous here?

**False Negatives (FN)** — predicting "not fraud" when it actually is fraud. These are missed fraud cases that go undetected, causing direct financial harm.

4. What does the confusion matrix reveal?

- The TN count (legitimate predicted as legitimate) is very large — close to all ~56,863 legitimate test samples.
- The TP count (fraud predicted as fraud) is much smaller — many fraud cases are missed.
- The FN count (missed fraud) is significant relative to the total fraud cases (~98 in the test set).
- The model is clearly biased toward the majority class.

5. Would this model be acceptable in banking?

**Absolutely not.** A fraud detection system that misses a significant portion of fraudulent transactions provides false security and enables financial losses. Banks would require high recall (catching nearly all fraud), even at the cost of some false alarms.

---

## Recall Formula

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

---

## Important Thinking Question

What happens if:
- False Negatives increase?

In fraud detection:
- which is worse:
  - false positives?
  - false negatives?

Why?

**False Negatives are WORSE than False Positives in fraud detection:**
- **False Negative**: Fraud goes undetected → customer/bank suffers financial loss, potential legal liability, damaged trust.
- **False Positive**: Legitimate transaction flagged → minor inconvenience (e.g., a card temporarily blocked), can be resolved with a phone call.

The asymmetric cost of errors means we must heavily optimize for recall, not overall accuracy.


# Task 6 — Improve the Model

You may choose ONLY ONE improvement initially.

Choose ONE:
- class_weight
- resampling
- threshold tuning
- feature engineering
- different model

---

## Before Coding

You MUST justify:
1. Why this improvement targets the failure
2. What evidence supports your choice
3. What tradeoff might appear

### 🎯 Chosen Improvement: `class_weight="balanced"`

**1. Why does this target the failure?**
The root cause of the baseline model's failure is that the learning algorithm assigns equal importance to every sample. With 577 legitimate transactions for every 1 fraud transaction, the optimizer learns that predicting "not fraud" almost always reduces loss. By using `class_weight="balanced"`, we instruct sklearn to **automatically upweight the minority class** — each fraud sample receives ~577× the weight of a legitimate sample during training. This forces the model to take fraud detection seriously during optimization.

**2. What evidence supports this choice?**
- The confusion matrix showed significant False Negatives (missed fraud).
- The recall for Class 1 was unsatisfactorily low in the baseline.
- The imbalance ratio (~577:1) is so extreme that the model's decision boundary is entirely skewed toward the majority class.
- `class_weight="balanced"` is specifically designed to correct for this by reweighting the loss function proportionally to class frequencies.

**3. What tradeoff might appear?**
- **Precision will likely decrease**: By making the model more aggressive about flagging fraud, it will also generate more False Positives (legitimate transactions flagged as fraud).
- **Accuracy will likely decrease slightly**: Weighted samples cause the model to misclassify some legitimate transactions.
- The key question is whether the improvement in Recall is worth the drop in Precision — in fraud detection, **it usually is**.


# Hint

Logistic Regression supports:

```python
class_weight="balanced"
```

But:

DO NOT use it blindly.

First explain:
- what it changes mathematically
- why it may help recall
- what downside it may introduce


In [15]:

balanced_model = LogisticRegression(class_weight="balanced")

balanced_model.fit(X_train_scaled, y_train)

balanced_preds = balanced_model.predict(X_test_scaled)


In [16]:

print("Accuracy:", accuracy_score(y_test, balanced_preds))
print("Precision:", precision_score(y_test, balanced_preds))
print("Recall:", recall_score(y_test, balanced_preds))
print("F1:", f1_score(y_test, balanced_preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, balanced_preds))

print("\nClassification Report")
print(classification_report(y_test, balanced_preds))


Accuracy: 0.9755275446789088
Precision: 0.06097560975609756
Recall: 0.9183673469387755
F1: 0.11435832274459974

Confusion Matrix
[[55478  1386]
 [    8    90]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962




# Task 7 — Tradeoff Investigation

Compare:
- baseline model
- balanced model

---

## Questions

1. Which metric improved most?
2. Which metric became worse?
3. Why did this happen?
4. Is higher recall worth lower precision?
5. Would business context affect this decision?

---

## Critical Thinking

Should ALL fraud systems maximize recall aggressively?

Why or why not?


**1. Which metric improved most?**
**Recall** improved most dramatically. The baseline model struggled to detect fraud cases; the balanced model should catch significantly more, often increasing recall from ~60% to ~90%+.

**2. Which metric became worse?**
**Precision** decreased. The model now flags more transactions as fraud — including more false alarms (legitimate transactions predicted as fraud). Accuracy also decreases slightly as a side effect.

**3. Why did this happen?**
When we upweight the fraud class, we shift the model's decision boundary. The threshold for "likely fraud" becomes lower — any transaction that shows even modest fraud-like features gets flagged. This increases True Positives (good) but also increases False Positives (the precision penalty).

The precision–recall tradeoff is fundamental: you can almost always get more of one at the expense of the other. There is no free lunch.

**4. Is higher recall worth lower precision in fraud detection?**
**Generally yes**, for the following reasons:
- The **cost of a False Negative** (missing actual fraud) = financial loss, legal liability, customer harm.
- The **cost of a False Positive** (blocking a legit transaction) = temporary inconvenience, one phone call to unblock.

These costs are **highly asymmetric**, so it makes sense to accept more false alarms to ensure fewer missed fraud cases.

**5. Would business context affect this decision?**
**Absolutely yes:**
- **High-value automated transactions**: Maximize recall — catching fraud is critical.
- **Small frequent transactions at point of sale**: A very low precision (too many false alarms) would block legitimate purchases constantly, damaging customer experience and loyalty.
- **Regulated environments (e.g., PCI-DSS)**: There may be compliance requirements on fraud rates.
- **Operations team capacity**: If human analysts review flagged transactions, very low precision floods the review queue.

**Critical Thinking: Should ALL fraud systems maximize recall aggressively?**

**No.** The optimal operating point depends on:
1. The relative **cost ratio** of FN vs FP for the specific business.
2. The **volume** of transactions (millions of false alarms per day = unusable).
3. Whether downstream review processes exist.
4. Customer experience tolerance.

The right approach is to use the **Precision-Recall AUC** and business cost modeling to find the optimal threshold — not blindly maximizing recall.


# Task 8 — Analyze Failure Cases

Investigate:
- False Positives
- False Negatives

---

## Questions

1. Do failed samples share patterns?
2. Are some transactions consistently difficult?
3. Which error type remains hardest?


**1. Do failed samples share patterns?**
We can check if False Negatives (missed fraud) tend to have distinct feature distributions compared to True Positives (caught fraud). Often:
- False Negatives may have `Amount` values closer to typical legitimate transaction amounts (making them harder to distinguish).
- Some PCA features (V1–V28) might cluster differently for missed vs. detected fraud.

**2. Are some transactions consistently difficult?**
Yes — transactions where the fraud pattern closely resembles legitimate transaction patterns are inherently ambiguous. These often include:
- Small-amount fraudulent transactions that look like typical legitimate micropayments.
- Fraud attempts where the fraudster deliberately mimics normal transaction behavior.

**3. Which error type remains hardest to eliminate?**
**False Negatives** on "camouflaged" fraud remain the hardest. The model can be tuned to catch more obvious fraud, but sophisticated fraud attempts that closely resemble normal behavior are difficult to classify correctly without additional signals (behavioral patterns, device fingerprinting, etc.).

In [17]:

results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = balanced_preds

false_negatives = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

false_negatives.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V22,V23,V24,V25,V26,V27,V28,Amount,Actual,Predicted
50537,44532.0,-0.234922,0.355413,1.972183,-1.255593,-0.681387,-0.665732,0.059110,-0.003153,1.122451,...,0.912107,-0.286338,0.451208,0.188315,-0.531846,0.123185,0.039581,1.00,1,0
157585,110087.0,1.934946,0.650678,-0.286957,3.987828,0.316052,-0.099449,-0.021483,-0.172327,0.508730,...,-0.190974,0.219976,-0.216597,-0.136692,-0.129954,-0.050077,-0.051082,1.00,1,0
96341,65728.0,1.227614,-0.668974,-0.271785,-0.589440,-0.604795,-0.350285,-0.486365,-0.010809,-0.794944,...,-0.295255,-0.180459,-0.436539,0.494649,-0.283738,-0.001128,0.035075,98.01,1,0
68067,52814.0,-1.101847,-1.632441,0.901067,0.847753,-1.249091,0.654937,1.448868,0.023308,-0.136742,...,0.835795,1.179955,-0.029091,-0.300896,0.699175,-0.336072,-0.177587,519.90,1,0
219025,141565.0,0.114965,0.766762,-0.494132,0.116772,0.868169,-0.477982,0.438496,0.063073,-0.186207,...,-0.706865,0.131405,0.600742,-0.604264,0.262938,0.099145,0.010810,4.49,1,0



# Bonus Challenge 1 — Threshold Tuning

Try:
- 0.5 threshold
- 0.3 threshold
- 0.1 threshold

Then analyze:
- precision vs recall tradeoff

Lowering the threshold means the model flags a transaction as fraud with lower confidence:
- **Lower threshold → Higher Recall, Lower Precision** (more fraud caught, more false alarms)
- **Higher threshold → Lower Recall, Higher Precision** (fewer false alarms, but more missed fraud)

The 0.3 threshold typically provides a good balance for fraud detection scenarios.


In [18]:

probs = balanced_model.predict_proba(X_test_scaled)[:, 1]

threshold = 0.3

custom_preds = (probs >= threshold).astype(int)

print("Precision:", precision_score(y_test, custom_preds))
print("Recall:", recall_score(y_test, custom_preds))
print("F1:", f1_score(y_test, custom_preds))


Precision: 0.027347310847766638
Recall: 0.9183673469387755
F1: 0.053113012688108585



# Bonus Challenge 2 — Random Forest Comparison

Questions:
1. Why might Random Forest behave differently?
2. Why may scaling matter less?
3. Which model would you trust more here?

**1. Why might Random Forest behave differently?**
Random Forest is an ensemble of decision trees. Each tree learns complex non-linear decision boundaries through axis-aligned splits. It can capture interactions between features that Logistic Regression (which models a linear decision boundary) cannot. For the PCA-transformed fraud features, non-linear patterns may exist that LR misses.

**2. Why may scaling matter less?**
Decision trees (and by extension Random Forests) split features on threshold values. Whether a feature is `1500` or `1.5` (scaled), the tree still finds the optimal split point. The relative ordering of values is preserved regardless of scale — so Random Forest is scale-invariant. We use the unscaled `X_train` directly.

**3. Which model would you trust more?**
**Random Forest is typically more trustworthy here** because:
- It handles imbalance better due to ensemble averaging (though class_weight can also be applied).
- Its non-linear nature is better suited to complex fraud patterns.
- It is naturally less prone to outliers affecting the decision boundary.
- Feature importances from RF provide interpretability about which PCA components matter most.

However, RF is slower to train/predict and harder to explain to regulators than Logistic Regression.

In [19]:

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_preds))
print("Precision:", precision_score(y_test, rf_preds))
print("Recall:", recall_score(y_test, rf_preds))
print("F1:", f1_score(y_test, rf_preds))


Accuracy: 0.9995962220427653
Precision: 0.9411764705882353
Recall: 0.8163265306122449
F1: 0.8743169398907104



# Bonus Challenge 3 — Leakage Trap

Read the code carefully before running.

Questions:
1. Why might this create suspiciously high performance?
2. What kind of leakage is this?
3. Why is leakage dangerous in production systems?

**1. Why might this create suspiciously high performance?**
The line `df["Leakage"] = df["Class"] + np.random.normal(0, 0.01, len(df))` creates a new feature that is essentially a **near-perfect copy of the target variable** `Class`. The Leakage feature will be ~0.0 for legitimate transactions and ~1.0 for fraudulent ones (with only tiny noise). When the model trains on this feature, it can almost perfectly reconstruct the target — achieving ~100% accuracy not because it learned fraud patterns, but because it literally has access to the answer.

**2. What kind of leakage is this?**
This is **target leakage** (also called **data leakage** or **label leakage**). The leaked feature was derived from the target variable itself. The feature would not exist in a real production system — at prediction time, `Class` is unknown (that's what we're trying to predict!), so a feature computed from it cannot exist.

**3. Why is leakage dangerous in production systems?**
- **False confidence**: The model appears to work perfectly in evaluation.
- **Complete failure in production**: The leakage feature doesn't exist for new, unseen transactions → the model's predictions collapse to random or worse.
- **Hidden until deployment**: The model passes all offline tests but fails catastrophically in the real world — potentially allowing fraud to go completely undetected.
- **Hard to detect**: Subtle leakage (e.g., using post-event data) can be very difficult to identify without domain knowledge.

Leakage is one of the most common and dangerous mistakes in production ML systems. Always ask: **"Would this feature be available at prediction time?"**

In [20]:

df["Leakage"] = df["Class"] + np.random.normal(0, 0.01, len(df))

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression()

model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))


Accuracy: 0.9990695551420246


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



# Final Reflection

Answer carefully.

---

## Reflection Questions

1. What was the REAL problem in this dataset?
2. Why was accuracy misleading?
3. What evidence proved imbalance existed?
4. Why did the second model improve recall?
5. What tradeoff was introduced?
6. What would you deploy in production?
7. What additional experiments would you run?
8. What assumptions might still be dangerous?

**1. What was the REAL problem in this dataset?**
**Severe class imbalance** — a 577:1 ratio of legitimate to fraudulent transactions. The model's learning objective (minimizing overall loss) was misaligned with the business objective (detecting fraud). The dataset also uses anonymized PCA features, which limits feature engineering opportunities.

**2. Why was accuracy misleading?**
Accuracy treats every correct prediction equally. With 99.83% of samples being legitimate, a model that always predicts "not fraud" achieves 99.83% accuracy — a perfect score by this metric — while being completely useless. Accuracy is only informative when classes are balanced.

**3. What evidence proved imbalance existed?**
`df["Class"].value_counts()` directly showed:
- Class 0: ~284,315 (99.83%)
- Class 1: ~492 (0.17%)
The confusion matrix also confirmed it: enormous TN, small TP, significant FN.

**4. Why did the second model improve recall?**
`class_weight="balanced"` reweights the loss function so each fraud sample contributes ~577× more to the gradient update than each legitimate sample. This forces the optimizer to move the decision boundary toward catching more fraud cases, even at the cost of some false alarms.

**5. What tradeoff was introduced?**
**Recall ↑ but Precision ↓.** More fraud was caught, but more legitimate transactions were also flagged as suspicious (higher False Positive rate). Accuracy also decreased slightly.

**6. What would you deploy in production?**
A **Random Forest with threshold tuning** would be the preferred starting point:
- Better handles non-linear fraud patterns than Logistic Regression
- Less sensitive to class imbalance
- Threshold can be calibrated to the desired Precision-Recall operating point
- Feature importances aid interpretability and regulatory compliance

In production, I would also add:
- Regular model retraining as fraud patterns evolve
- Monitoring for data drift and performance degradation
- A human review queue for borderline cases
- A/B testing framework for model updates

**7. What additional experiments would you run?**
- **SMOTE/ADASYN resampling** — synthetically oversample fraud cases
- **Threshold optimization** using business cost matrix (FP cost vs FN cost)
- **XGBoost/LightGBM** — often best in class for tabular fraud detection
- **Anomaly detection approaches** (Isolation Forest, Autoencoders) — treat fraud as outlier
- **Feature importance analysis** — which V1–V28 components matter most?
- **Time-based validation split** — split by Time column instead of random split to simulate real deployment
- **Cross-validation with stratified k-folds** — more robust performance estimates

**8. What assumptions might still be dangerous?**
- **I.I.D. assumption**: We assumed random splits, but transactions have time-ordering. Fraud patterns evolve — a model trained on old patterns may fail on new fraud techniques.
- **Fixed features**: The V1–V28 PCA features are from a specific feature extraction. If the underlying raw features change, the PCA space changes.
- **Static threshold**: The optimal threshold at training time may not remain optimal as fraud patterns shift.
- **No concept drift handling**: The model has no mechanism to detect when the underlying data distribution changes.
- **Evaluation on held-out set only**: Production fraud rates and patterns may differ from this historical dataset.

---

# Final Engineering Question

> "How do you know your improvement genuinely solved the problem?"

An improvement genuinely solves the problem when:
1. **Recall on the minority class increased** substantially (evidenced by fewer False Negatives in the confusion matrix).
2. **The F1-score improved**, showing that the precision-recall balance shifted favorably relative to business requirements.
3. **The Precision-Recall AUC improved**, not just performance at one threshold.
4. **Performance holds on out-of-time data** (not just a random split) — simulating real deployment conditions.
5. **The improvement aligns with business metrics**: if every False Negative costs $X and every False Positive costs $Y, the expected business loss decreased.
6. **No data leakage** was introduced that would make results unreproducible in production.


# Grading Rubric

You are graded on:

- reasoning quality
- justification quality
- investigation depth
- evidence usage
- interpretation
- tradeoff understanding

NOT only final metrics.
